In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
from src import config

In [3]:
import hopsworks

# connect to the project
project = hopsworks.login(
    project=config.HOPSWORKS_PROJECT_NAME,
    api_key_value=config.HOPSWORKS_API_KEY,
    cert_folder="./hopsworks-certs",
)

# connect to feature store
feature_store = project.get_feature_store()

# connect to feature group 
feature_group = feature_store.get_feature_group(
    name=config.FEATURE_GROUP_NAME,
    version=config.FEATURE_GROUP_VERSION,
)

2026-06-25 14:01:15,133 INFO: Initializing external client
2026-06-25 14:01:15,133 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-25 14:01:17,423 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960


In [4]:
feature_group.read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (26.17s) 


,pickup_hour,rides,pickup_location_id
0,2025-10-13 06:00:00+00:00,27,249
1,2026-02-15 16:00:00+00:00,24,209
2,2026-02-01 06:00:00+00:00,0,172
3,2025-06-02 00:00:00+00:00,6,36
4,2025-12-01 01:00:00+00:00,7,137
...,...,...,...
3230198,2025-03-15 00:00:00+00:00,1,63
3230199,2026-04-04 01:00:00+00:00,0,109
3230200,2026-01-27 22:00:00+00:00,22,74
3230201,2026-03-01 06:00:00+00:00,10,7


In [5]:
# Create feature view if it doesn't exists yet
# This feature view uses only on feature group, so the query is trival
try:
    feature_store.create_feature_view(
        name=config.FEATURE_VIEW_NAME,
        version=config.FEATURE_VIEW_VERSION,
        query=feature_group.select_all(),
    )
except:
    print(f"Feature view already existed, Skip creation.")

# Get feature view
feature_view = feature_store.get_feature_view(
        name=config.FEATURE_VIEW_NAME,
        version=config.FEATURE_VIEW_VERSION,
    )


Feature view already existed, Skip creation.


In [6]:
ts_data, _ = feature_view.training_data(
    description="Time-series hourly taxi rides",
)

2026-06-25 14:02:22,532 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (29.90s) 


In [7]:
ts_data.sort_values(by=['pickup_location_id','pickup_hour'], inplace=True)
ts_data

,pickup_hour,rides,pickup_location_id
3006989,2025-01-01 00:00:00+00:00,0,1
2360573,2025-01-01 01:00:00+00:00,0,1
2045466,2025-01-01 02:00:00+00:00,0,1
1691654,2025-01-01 03:00:00+00:00,0,1
1888190,2025-01-01 04:00:00+00:00,0,1
...,...,...,...
467719,2026-06-25 04:00:00+00:00,0,265
470619,2026-06-25 05:00:00+00:00,0,265
470123,2026-06-25 06:00:00+00:00,5,265
467914,2026-06-25 07:00:00+00:00,2,265


In [8]:
from src.data import transform_ts_data_info_feature_and_target

features, targets = transform_ts_data_info_feature_and_target(
    ts_data,
    input_seq_len=24*29, # one month
    step_size=23,
)

features_and_target = features.copy()
features_and_target['target_rides_next_hour'] = targets

print(f'{features_and_target=}')

100%|██████████| 262/262 [01:41<00:00,  2.58it/s]


features_and_target=        rides_previous_696_hours  rides_previous_695_hours  \
0                            0.0                       0.0   
1                            0.0                       0.0   
2                            0.0                       0.0   
3                            2.0                       0.0   
4                            0.0                       1.0   
...                          ...                       ...   
132736                       3.0                       1.0   
132737                       4.0                       1.0   
132738                       3.0                       1.0   
132739                       1.0                       4.0   
132740                       1.0                       2.0   

        rides_previous_694_hours  rides_previous_693_hours  \
0                            0.0                       0.0   
1                            0.0                       0.0   
2                            0.0                 

In [9]:
print(features_and_target['pickup_hour'].dtype)
print(features_and_target[['pickup_hour']].head())

object
                 pickup_hour
0  2025-01-30 00:00:00+00:00
1  2025-01-30 23:00:00+00:00
2  2025-01-31 22:00:00+00:00
3  2025-02-01 21:00:00+00:00
4  2025-02-02 20:00:00+00:00


In [10]:
from datetime import date, timedelta
from pytz import timezone
import pandas as pd
from src.data_split import train_test_split

cutoff_date = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=28)

print(f'{cutoff_date=}')

X_train, y_train, X_test, y_test = train_test_split(
    features_and_target,
    cutoff_date,
    target_colum_name='target_rides_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')


cutoff_date=Timestamp('2026-05-28 11:04:37.519387+0000', tz='UTC')
X_train.shape=(125230, 698)
y_train.shape=(125230,)
X_test.shape=(7511, 698)
y_test.shape=(7511,)


In [11]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    hyperparams = {
        "metric": "mae",
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 100),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    tss = TimeSeriesSplit(n_splits=4)
    scores = []

    for train_index, val_index in tss.split(X_train):

        X_train_ = X_train.iloc[train_index, :].copy()
        X_val_ = X_train.iloc[val_index, :].copy()

        y_train_ = y_train.iloc[train_index].copy()
        y_val_ = y_train.iloc[val_index].copy()

        pipeline = get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)

        y_pred = pipeline.predict(X_val_)
        mae = mean_absolute_error(y_val_, y_pred)

        scores.append(mae)

    return np.array(scores).mean()


In [12]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)

[I 2026-06-25 14:04:40,047] A new study created in memory with name: no-name-ba9ff020-de69-4f11-ac56-82f7c493e5f7
[I 2026-06-25 14:05:49,518] Trial 0 finished with value: 4.582578288193356 and parameters: {'num_leaves': 249, 'feature_fraction': 0.6013827841840604, 'bagging_fraction': 0.48385454360882874, 'min_child_samples': 53, 'n_estimators': 424, 'learning_rate': 0.1893739361253429, 'max_depth': 10, 'reg_alpha': 1.4838223753516779e-05, 'reg_lambda': 0.004080034947980337}. Best is trial 0 with value: 4.582578288193356.
[I 2026-06-25 14:06:14,902] Trial 1 finished with value: 6.17349082070223 and parameters: {'num_leaves': 240, 'feature_fraction': 0.8922024994636635, 'bagging_fraction': 0.6987235565861685, 'min_child_samples': 66, 'n_estimators': 173, 'learning_rate': 0.011780197494468042, 'max_depth': 3, 'reg_alpha': 3.8385037290753594, 'reg_lambda': 0.0017771188921746196}. Best is trial 0 with value: 4.582578288193356.
[I 2026-06-25 14:07:49,133] Trial 2 finished with value: 4.40029

In [13]:
best_params = study.best_trial.params
print(f'{best_params=}')

best_params={'num_leaves': 130, 'feature_fraction': 0.2622501047068541, 'bagging_fraction': 0.9948789323285829, 'min_child_samples': 15, 'n_estimators': 934, 'learning_rate': 0.07473981124675212, 'max_depth': 12, 'reg_alpha': 9.669526816894104e-08, 'reg_lambda': 0.00021869692287823392}


In [14]:
pipeline = get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeaturesengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](698,)","['rides_previous_696_hours','rides_previous_695_hours', 'rides_previous_694_hours',...,'rides_previous_1_hours','pickup_hour', 'pickup_location_id']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,698
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...001E833D00040>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [15]:
prediction = pipeline.predict(X_test)
test_mae = mean_absolute_error(y_test, prediction)
print(f'{test_mae=:.4f}')

test_mae=6.8883


In [16]:
from pathlib import Path

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

print(MODELS_DIR.resolve())


D:\My-Github\taxi_demand_predictor\notebooks\models


In [17]:
import joblib

joblib.dump(pipeline, MODELS_DIR / "model.pkl")

['models\\model.pkl']

In [18]:
from hsml.schema import Schema
from hsml.model_schema import ModelSchema

input_schema = Schema(X_train)
output_schema = Schema(y_train)
model_schema = ModelSchema(input_schema=input_schema, output_schema=output_schema)


In [19]:
model_registy = project.get_model_registry()

model = model_registy.sklearn.create_model(
    name="taxi_demand_predictor_next_hour",
    metrics={
        "test_mae": test_mae
    },
    description="LIGHTGBM Regressor with a bit of hyper-parameter tuning",
    input_example=X_train.sample(),
    model_schema=model_schema
)

model.save( MODELS_DIR / 'model.pkl')

  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'models\model.pkl' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading d:\My-Github\taxi_demand_predictor\notebooks\models\model.pkl: 0.000%|          | 0/10504707 elapsed…

Uploading C:\Users\AHMEDS~1\AppData\Local\Temp\tmpq7tiba7i\input_example.json: 0.000%|          | 0/3561 elaps…

Uploading C:\Users\AHMEDS~1\AppData\Local\Temp\tmpq7tiba7i\model_schema.json: 0.000%|          | 0/64140 elaps…

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/34960/models/taxi_demand_predictor_next_hour/2


Model(name: 'taxi_demand_predictor_next_hour', version: 2)